
## Step 1: Mount Google Drive

In [11]:
from google.colab import drive
import os

# Mount Google Drive
drive.mount('/content/drive')

print("\n✓ Google Drive mounted successfully!")
print("\nYour files are now accessible at: /content/drive/MyDrive/")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

✓ Google Drive mounted successfully!

Your files are now accessible at: /content/drive/MyDrive/


## Step 2: Import Required Libraries

In [12]:
import re
import numpy as np
from collections import defaultdict
from math import log

print("✓ All libraries imported successfully!")

✓ All libraries imported successfully!


## Step 3: Define All Functions

In [13]:
import re
import numpy as np
from collections import defaultdict
from math import log

# Preprocessing function
def preprocess(text):
    return re.findall(r'\b\w+\b', text.lower())

# Load documents
def load_documents(folder_path):
    docs = {}
    for filename in os.listdir(folder_path):
        if filename.endswith('.txt'):
            with open(os.path.join(folder_path, filename), 'r', encoding='utf-8') as file:
                docs[filename] = preprocess(file.read())
    print(f"Loaded {len(docs)} documents")
    return docs

# Load queries
def load_queries(query_file_path):
    with open(query_file_path, 'r', encoding='utf-8') as file:
        queries = [line.strip() for line in file.readlines()]
    print(f"Loaded {len(queries)} queries")
    return queries

# Compute term frequencies and document frequencies
def compute_statistics(docs):
    doc_count = len(docs)
    term_doc_freq = defaultdict(int)
    term_freq = defaultdict(lambda: defaultdict(int))

    for doc_id, words in docs.items():
        word_set = set(words)
        for word in words:
            term_freq[doc_id][word] += 1
        for word in word_set:
            term_doc_freq[word] += 1

    return term_freq, term_doc_freq, doc_count

# Compute relevance probabilities using BIM
def compute_relevance_prob(query, term_freq, term_doc_freq, doc_count):
    scores = {}
    for doc_id in term_freq:
        score = 1.0
        for term in query:
            tf = term_freq[doc_id].get(term, 0)
            df = term_doc_freq.get(term, 0)
            # Laplace smoothing for probabilities
            p_term_given_relevant = (tf + 1) / (sum(term_freq[doc_id].values()) + len(term_doc_freq))
            p_term_given_not_relevant = (df + 1) / (doc_count - df + len(term_doc_freq))
            score *= (p_term_given_relevant / p_term_given_not_relevant)
        scores[doc_id] = score
    return scores

# Main retrieval function
def retrieve_documents(folder_path, query_file_path, top_k=5, save_to_file=True, output_file_path='/content/drive/MyDrive/BIM_results.txt'):
    print("="*80)
    print("Binary Independence Model (BIM) - Document Retrieval")
    print("="*80)
    print()

    docs = load_documents(folder_path)
    queries = load_queries(query_file_path)

    term_freq, term_doc_freq, doc_count = compute_statistics(docs)
    print(f"✓ Total unique terms: {len(term_doc_freq)}")
    print()
    print("="*80)
    print()

    # Prepare output file if needed
    output_lines = []

    for idx, query in enumerate(queries, 1):
        query_terms = preprocess(query)
        if not query_terms:
            continue

        scores = compute_relevance_prob(query_terms, term_freq, term_doc_freq, doc_count)
        ranked_docs = sorted(scores.items(), key=lambda item: item[1], reverse=True)

        # Print to console
        output = f"Query {idx}: {query}\n"
        output += f"Query terms: {query_terms}\n\n"
        output += f"Top {top_k} Results:\n"
        output += "-"*80 + "\n"

        for rank, (doc_id, score) in enumerate(ranked_docs[:top_k], 1):
            output += f"  {rank}. {doc_id:35s} Score: {score:.6f}\n"

        output += "\n" + "="*80 + "\n\n"

        print(output)
        output_lines.append(output)

    # Save to file if requested
    if save_to_file:
        with open(output_file_path, 'w', encoding='utf-8') as f:
            f.write(f"Binary Independence Model Results\n")
            f.write(f"Total Documents: {doc_count}\n")
            f.write(f"Total Queries: {len(queries)}\n")
            f.write(f"Total Unique Terms: {len(term_doc_freq)}\n")
            f.write("="*80 + "\n\n")
            f.writelines(output_lines)
        print(f"\n✓ Results saved to: {output_file_path}")

    print(f"\n✓ Processed {len(queries)} queries")
    print(f"✓ Ranked {doc_count} documents")


## Step 5: Set Paths



In [14]:
# Update these paths according to your Google Drive structure
folder_path = '/content/drive/MyDrive/Trump Speechs'
query_file_path = '/content/drive/MyDrive/queries1.txt'
# Verify paths exist
print("Checking if paths exist...\n")

if os.path.exists(folder_path):
    txt_files = [f for f in os.listdir(folder_path) if f.endswith('.txt')]
    print(f"✓ Folder found: {folder_path}")
    print(f"  Contains {len(txt_files)} .txt files")
else:
    print(f"✗ Folder NOT found: {folder_path}")
    print("  Please update the folder_path variable above")

print()

if os.path.exists(query_file_path):
    print(f"✓ Query file found: {query_file_path}")
else:
    print(f"✗ Query file NOT found: {query_file_path}")
    print("  Please update the query_file_path variable above")

Checking if paths exist...

✓ Folder found: /content/drive/MyDrive/Trump Speechs
  Contains 56 .txt files

✓ Query file found: /content/drive/MyDrive/queries1.txt


In [15]:
# Run the BIM model
retrieve_documents(
    folder_path=folder_path,
    query_file_path=query_file_path,
    top_k=5,
    save_to_file=True,
    output_file_path='/content/BIM_results_colab.txt'
)

Binary Independence Model (BIM) - Document Retrieval

Loaded 56 documents
Loaded 30 queries
✓ Total unique terms: 7040


Query 1: to
Query terms: ['to']

Top 5 Results:
--------------------------------------------------------------------------------
  1. speech_8.txt                        Score: 1.989384
  2. speech_19.txt                       Score: 1.988463
  3. speech_7.txt                        Score: 1.503374
  4. speech_3.txt                        Score: 1.464820
  5. speech_30.txt                       Score: 1.377421



Query 2: america strong
Query terms: ['america', 'strong']

Top 5 Results:
--------------------------------------------------------------------------------
  1. speech_2.txt                        Score: 0.026096
  2. speech_7.txt                        Score: 0.023406
  3. speech_13.txt                       Score: 0.012004
  4. speech_51.txt                       Score: 0.010722
  5. speech_36.txt                       Score: 0.008305



Query 3: to bring 